In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
import json
import time

# O objeto 'spark' já vem instanciado no ambiente Databricks
print("Sessão Spark ativa:", spark.version)

In [0]:
# Definindo os dados simulados diretamente em memória para o teste inicial
# (posteriormente podemos apontar para spark.read.csv() dos seus arquivos do Git)

schema_template = StructType([
    StructField("id_cenario", IntegerType(), True),
    StructField("chave_roteiro", StringType(), True),
    StructField("conta_debito_esperada", StringType(), True),
    StructField("conta_credito_esperada", StringType(), True),
    StructField("valor_esperado", DoubleType(), True)
])

schema_snapshot = StructType([
    StructField("id_cenario", IntegerType(), True),
    StructField("chave_roteiro", StringType(), True),
    StructField("conta_debito_gerada", StringType(), True),
    StructField("conta_credito_gerada", StringType(), True),
    StructField("valor_gerado", DoubleType(), True)
])

# Para simular os 500 cenários de teste
# Criamos um DataFrame sintético com Spark
dados_template = [(i, f"ROT_{i:04d}", "1.1.1.01", "2.1.1.01", 1500.0) for i in range(1, 501)]
df_template = spark.createDataFrame(dados_template, schema_template)

# Simulando 450 registros gerados (gerando 50 órfãos e alguns divergentes)
dados_snapshot = []
for i in range(1, 451):
    # Injetando 23 divergências controladas
    if i <= 23:
        dados_snapshot.append((i, f"ROT_{i:04d}", "1.1.9.99", "2.1.1.01", 1500.0)) # Conta divergente
    else:
        dados_snapshot.append((i, f"ROT_{i:04d}", "1.1.1.01", "2.1.1.01", 1500.0)) # Sucesso

df_snapshot = spark.createDataFrame(dados_snapshot, schema_snapshot)

print(f"Template carregado: {df_template.count()} registros")
print(f"Snapshot carregado: {df_snapshot.count()} registros")

In [0]:
inicio = time.time()

# 1. Left Join entre Template (expectativa) e Snapshot (processado)
df_join = df_template.alias("t").join(
    df_snapshot.alias("s"),
    on="id_cenario",
    how="left"
)

# 2. Regra de Negócio: Classificação em 3 status determinísticos
df_conciliado = df_join.withColumn(
    "status_homologacao",
    F.when(F.col("s.id_cenario").isNull(), "Não Sensibilizado")
     .when(
         (F.col("t.conta_debito_esperada") != F.col("s.conta_debito_gerada")) | 
         (F.col("t.conta_credito_esperada") != F.col("s.conta_credito_gerada")) |
         (F.col("t.valor_esperado") != F.col("s.valor_gerado")), 
         "Divergente"
     )
     .otherwise("Sensibilizado com Sucesso")
)

# No Serverless Compute, removemos o df_conciliado.cache()
total_processado = df_conciliado.count()
tempo_execucao = time.time() - inicio

print(f"Conciliação finalizada em {tempo_execucao:.4f} segundos")

In [0]:
# Agrupamento e cálculo de proporções
resumo_status = df_conciliado.groupBy("status_homologacao").count() \
    .withColumn("percentual", F.round((F.col("count") / total_processado) * 100, 2))

display(resumo_status)

In [0]:
# Filtra amostras das anomalias para fornecer contexto ao Agente
divergencias_sample = df_conciliado.filter(F.col("status_homologacao") != "Sensibilizado com Sucesso") \
    .select(
        "id_cenario", 
        "t.chave_roteiro", 
        "status_homologacao", 
        "t.conta_debito_esperada", 
        "s.conta_debito_gerada"
    ) \
    .limit(10) \
    .toPandas() \
    .to_dict(orient="records")

# Dicionário consolidado de contagem
metricas_dict = {row["status_homologacao"]: row["count"] for row in resumo_status.collect()}

payload_agente = {
    "total_cenarios": total_processado,
    "metricas": metricas_dict,
    "amostras_anomalias": divergencias_sample
}

import json
print("Payload estruturado para a Camada Cognitiva:")
print(json.dumps(payload_agente, indent=2, ensure_ascii=False))

In [0]:
# EXECUTAR UMA VEZ: Adiciona sua chave API do Gemini ao scope existente
# A chave não será exibida na tela nem salva no notebook

from databricks.sdk import WorkspaceClient
import getpass

print("📋 Instruções:")
print("1. Obtenha sua chave API em: https://ai.google.dev/gemini-api/docs/api-key")
print("2. Cole a chave abaixo quando solicitado (ela ficará oculta)")
print("3. Pressione Enter\n")

# Solicita a chave de forma segura (não aparece enquanto digita)
api_key = getpass.getpass(prompt="Cole sua chave API do Gemini: ")

if api_key:
    print("\n⏳ Adicionando secret ao scope 'gemini-secrets'...")
    
    try:
        # Adiciona o secret usando a Databricks SDK
        w = WorkspaceClient()
        w.secrets.put_secret(
            scope="gemini-secrets",
            key="gemini_api_key",
            string_value=api_key
        )
        
        print("\n✅ Secret 'gemini_api_key' configurado com sucesso!")
        print("\n✨ Próximos passos:")
        print("   1. Execute a próxima célula para verificar (opcional)")
        print("   2. Execute a Cell 9 para usar o agente cognitivo")
    except Exception as e:
        print(f"\n❌ Erro ao configurar secret: {e}")
else:
    print("\n⚠️ Nenhuma chave fornecida. Execute novamente para tentar.")

In [0]:
from databricks.sdk import WorkspaceClient

# Lista todos os scopes de secrets existentes no workspace usando a SDK
w = WorkspaceClient()

try:
    scopes = w.secrets.list_scopes()
    print("✅ Scopes de secrets existentes no workspace:\n")
    
    scope_list = list(scopes)
    if scope_list:
        for scope in scope_list:
            print(f"  • Nome: {scope.name}")
            if scope.backend_type:
                print(f"    Tipo: {scope.backend_type}")
            print()
    else:
        print("  Nenhum scope encontrado.")
        print("  💡 Você precisa criar um scope primeiro na interface do Databricks.")
except Exception as e:
    print(f"❌ Erro ao listar scopes: {e}")

In [0]:
# OPCIONAL: Verifica se o secret foi criado com sucesso
# Você deve ver: gemini_api_key (o valor aparecerá mascarado)

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
secrets = w.secrets.list_secrets(scope="gemini-secrets")

print("✅ Secrets no scope 'gemini-secrets':\n")
for secret in secrets:
    print(f"  • {secret.key} (valor: **REDACTED**)")

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import json
import time
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
from google.genai.errors import ServerError

# 1. Recupera a chave da API do Gemini de forma segura via Databricks Secrets
GEMINI_API_KEY = dbutils.secrets.get(scope="gemini-secrets", key="gemini_api_key")

client = genai.Client(api_key=GEMINI_API_KEY)

# 2. Definição do Esquema de Retorno Estruturado
class RelatorioAuditoriaSOX(BaseModel):
    status_conformidade: str = Field(description="Classificação formal: 'Em Risco Material', 'Aprovado com Ressalvas' ou 'Conforme'")
    acuracia_auditada_pct: float = Field(description="Acurácia calculada com base no total e sucessos")
    diagnostico_causa_raiz: str = Field(description="Análise técnica concisa sobre os padrões observados nas anomalias")
    impacto_regulatorio_sox: str = Field(description="Avaliação de risco material sob a ótica da Seção 404 da Lei Sarbanes-Oxley")
    recomendacoes_mitigacao: list[str] = Field(description="Ações prescritivas para saneamento no motor de roteamento contábil")

# 3. Definição do System Prompt (Skill do Agente)
system_instruction = """
Você é um Auditor Digital de Sistemas Contábeis automatizado, atuando em conjunto com um pipeline de DataOps em ambiente Databricks.
Sua atribuição é avaliar sumários de homologação de motores de roteamento financeiro e gerar diagnósticos estruturados em estrita observância à Seção 404 da Lei Sarbanes-Oxley [SOX].
Seja estritamente técnico, objetivo, baseie-se unicamente nos dados numéricos e padrões de amostragem fornecidos e aponte os riscos de distorção material.
"""

# 4. Invocação da Camada Cognitiva
prompt_usuario = f"""
Avalie o seguinte payload gerado pela esteira de conciliação distribuída e emita o parecer formal:

{json.dumps(payload_agente, indent=2, ensure_ascii=False)}
"""

inicio_ia = time.time()

# Implementa retry com exponential backoff para lidar com erros 503
max_tentativas = 3
tentativa = 0
resposta = None

while tentativa < max_tentativas:
    try:
        print(f"🔄 Tentativa {tentativa + 1}/{max_tentativas}...")
        
        resposta = client.models.generate_content(
            model="models/gemini-3.6-flash",  # Nome completo do modelo
            contents=prompt_usuario,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.1,
                response_mime_type="application/json",
                response_schema=RelatorioAuditoriaSOX,
            ),
        )
        print("✅ Resposta recebida com sucesso!")
        break  # Sucesso, sai do loop
        
    except ServerError as e:
        tentativa += 1
        if tentativa < max_tentativas:
            tempo_espera = 2 ** tentativa  # Exponential backoff: 2, 4, 8 segundos
            print(f"⚠️ Erro 503: API temporariamente indisponível. Aguardando {tempo_espera}s...")
            time.sleep(tempo_espera)
        else:
            print("❌ Todas as tentativas falharam. O serviço Gemini está temporariamente indisponível.")
            raise

tempo_ia = time.time() - inicio_ia

# 5. Desserialização do resultado validado pelo Pydantic
relatorio_final: RelatorioAuditoriaSOX = resposta.parsed

print(f"Auditoria Cognitiva concluída em {tempo_ia:.2f} segundos\n")
print(json.dumps(relatorio_final.model_dump(), indent=2, ensure_ascii=False))

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

print("🧪 INICIANDO BATERIA DE TESTES DE PERFORMANCE")
print("=" * 60)
print(f"📅 Data/Hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔢 Número de execuções: 5")
print(f"📊 Cenários por execução: 500")
print("=" * 60)
print()

# Listas para armazenar os tempos de cada execução
tempos_spark = []
tempos_gemini = []
tempos_totais = []
resultados_execucoes = []

for i in range(1, 6):
    print(f"\n{'='*60}")
    print(f"🔄 EXECUÇÃO {i}/5")
    print("=" * 60)
    
    try:
        # ============================================
        # ETAPA 1: CONCILIAÇÃO SPARK
        # ============================================
        print("\n⚡ [1/2] Executando Conciliação Spark...")
        inicio_spark = time.time()
        
        # Left Join entre Template e Snapshot
        df_join = df_template.alias("t").join(
            df_snapshot.alias("s"),
            on="id_cenario",
            how="left"
        )
        
        # Classificação em 3 status
        df_conciliado = df_join.withColumn(
            "status_homologacao",
            F.when(F.col("s.id_cenario").isNull(), "Não Sensibilizado")
             .when(
                 (F.col("t.conta_debito_esperada") != F.col("s.conta_debito_gerada")) | 
                 (F.col("t.conta_credito_esperada") != F.col("s.conta_credito_gerada")) |
                 (F.col("t.valor_esperado") != F.col("s.valor_gerado")), 
                 "Divergente"
             )
             .otherwise("Sensibilizado com Sucesso")
        )
        
        total_processado = df_conciliado.count()
        tempo_spark = time.time() - inicio_spark
        tempos_spark.append(tempo_spark)
        
        print(f"   ✅ Conciliação concluída: {tempo_spark:.4f}s")
        
        # Preparar payload para Gemini
        resumo_status = df_conciliado.groupBy("status_homologacao").count() \
            .withColumn("percentual", F.round((F.col("count") / total_processado) * 100, 2))
        
        divergencias_sample = df_conciliado.filter(F.col("status_homologacao") != "Sensibilizado com Sucesso") \
            .select(
                "id_cenario", 
                "t.chave_roteiro", 
                "status_homologacao", 
                "t.conta_debito_esperada", 
                "s.conta_debito_gerada"
            ) \
            .limit(10) \
            .toPandas() \
            .to_dict(orient="records")
        
        metricas_dict = {row["status_homologacao"]: row["count"] for row in resumo_status.collect()}
        
        payload_agente = {
            "total_cenarios": total_processado,
            "metricas": metricas_dict,
            "amostras_anomalias": divergencias_sample
        }
        
        # ============================================
        # ETAPA 2: AUDITORIA COGNITIVA GEMINI
        # ============================================
        print("\n🤖 [2/2] Executando Auditoria Cognitiva Gemini...")
        inicio_gemini = time.time()
        
        prompt_usuario = f"""
Avalie o seguinte payload gerado pela esteira de conciliação distribuída e emita o parecer formal:

{json.dumps(payload_agente, indent=2, ensure_ascii=False)}
"""
        
        # Retry com exponential backoff
        max_tentativas = 3
        tentativa = 0
        resposta = None
        
        while tentativa < max_tentativas:
            try:
                resposta = client.models.generate_content(
                    model="models/gemini-3.6-flash",
                    contents=prompt_usuario,
                    config=types.GenerateContentConfig(
                        system_instruction=system_instruction,
                        temperature=0.1,
                        response_mime_type="application/json",
                        response_schema=RelatorioAuditoriaSOX,
                    ),
                )
                break
            except ServerError as e:
                tentativa += 1
                if tentativa < max_tentativas:
                    tempo_espera = 2 ** tentativa
                    print(f"   ⚠️  Retry {tentativa}/{max_tentativas-1} (aguardando {tempo_espera}s)...")
                    time.sleep(tempo_espera)
                else:
                    raise
        
        tempo_gemini = time.time() - inicio_gemini
        tempos_gemini.append(tempo_gemini)
        
        tempo_total = tempo_spark + tempo_gemini
        tempos_totais.append(tempo_total)
        
        print(f"   ✅ Auditoria concluída: {tempo_gemini:.4f}s")
        print(f"\n📊 RESULTADO EXECUÇÃO {i}:")
        print(f"   • Spark:  {tempo_spark:.4f}s")
        print(f"   • Gemini: {tempo_gemini:.4f}s")
        print(f"   • Total:  {tempo_total:.4f}s")
        
        # Armazenar resultado estruturado
        relatorio_final = resposta.parsed
        resultados_execucoes.append({
            "execucao": i,
            "tempo_spark": tempo_spark,
            "tempo_gemini": tempo_gemini,
            "tempo_total": tempo_total,
            "status_conformidade": relatorio_final.status_conformidade,
            "acuracia_pct": relatorio_final.acuracia_auditada_pct
        })
        
    except Exception as e:
        print(f"\n❌ ERRO na execução {i}: {e}")
        # Continua para próxima execução
        continue

print("\n" + "=" * 60)
print("✅ BATERIA DE TESTES CONCLUÍDA")
print("=" * 60)

In [0]:
# ============================================
# ANÁLISE ESTATÍSTICA DOS RESULTADOS
# ============================================

print("\n" + "📊 " + "=" * 58)
print("📊 RELATÓRIO ESTATÍSTICO DE PERFORMANCE")
print("=" * 60)

# Converter para arrays numpy para cálculos
array_spark = np.array(tempos_spark)
array_gemini = np.array(tempos_gemini)
array_total = np.array(tempos_totais)

# Calcular estatísticas descritivas
estatisticas = {
    "Spark": {
        "Média": np.mean(array_spark),
        "Mediana": np.median(array_spark),
        "Desvio Padrão": np.std(array_spark, ddof=1),
        "Mínimo": np.min(array_spark),
        "Máximo": np.max(array_spark),
        "Coef. Variação %": (np.std(array_spark, ddof=1) / np.mean(array_spark)) * 100
    },
    "Gemini": {
        "Média": np.mean(array_gemini),
        "Mediana": np.median(array_gemini),
        "Desvio Padrão": np.std(array_gemini, ddof=1),
        "Mínimo": np.min(array_gemini),
        "Máximo": np.max(array_gemini),
        "Coef. Variação %": (np.std(array_gemini, ddof=1) / np.mean(array_gemini)) * 100
    },
    "Total": {
        "Média": np.mean(array_total),
        "Mediana": np.median(array_total),
        "Desvio Padrão": np.std(array_total, ddof=1),
        "Mínimo": np.min(array_total),
        "Máximo": np.max(array_total),
        "Coef. Variação %": (np.std(array_total, ddof=1) / np.mean(array_total)) * 100
    }
}

# Exibir tabela de estatísticas
print("\n📈 ESTATÍSTICAS DESCRITIVAS (em segundos):\n")
df_stats = pd.DataFrame(estatisticas).T
print(df_stats.to_string(float_format="%.4f"))

# Exibir tabela de resultados individuais
print("\n\n📝 RESULTADOS INDIVIDUAIS POR EXECUÇÃO:\n")
df_resultados = pd.DataFrame(resultados_execucoes)
print(df_resultados.to_string(index=False, float_format="%.4f"))

# ============================================
# VISUALIZAÇÃO GRÁFICA
# ============================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('📊 Benchmark de Performance - Pipeline de Auditoria Cognitiva SOX', 
             fontsize=16, fontweight='bold', y=0.995)

# Gráfico 1: Tempos por Execução (Linhas)
ax1 = axes[0, 0]
execucoes = list(range(1, len(tempos_spark) + 1))
ax1.plot(execucoes, tempos_spark, marker='o', linewidth=2, markersize=8, label='Spark', color='#FF6B35')
ax1.plot(execucoes, tempos_gemini, marker='s', linewidth=2, markersize=8, label='Gemini', color='#004E89')
ax1.plot(execucoes, tempos_totais, marker='^', linewidth=2, markersize=8, label='Total', color='#1B9E77', linestyle='--')
ax1.set_xlabel('Execução', fontsize=11, fontweight='bold')
ax1.set_ylabel('Tempo (segundos)', fontsize=11, fontweight='bold')
ax1.set_title('Tempo de Execução por Etapa', fontsize=12, fontweight='bold', pad=10)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_xticks(execucoes)

# Gráfico 2: Comparação de Médias (Barras)
ax2 = axes[0, 1]
categorias = ['Spark', 'Gemini', 'Total']
medias = [estatisticas['Spark']['Média'], 
          estatisticas['Gemini']['Média'], 
          estatisticas['Total']['Média']]
desvios = [estatisticas['Spark']['Desvio Padrão'], 
           estatisticas['Gemini']['Desvio Padrão'], 
           estatisticas['Total']['Desvio Padrão']]

bars = ax2.bar(categorias, medias, yerr=desvios, capsize=10, 
               color=['#FF6B35', '#004E89', '#1B9E77'], alpha=0.8, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Tempo Médio (segundos)', fontsize=11, fontweight='bold')
ax2.set_title('Tempo Médio ± Desvio Padrão', fontsize=12, fontweight='bold', pad=10)
ax2.grid(True, alpha=0.3, linestyle='--', axis='y')

# Adicionar valores nas barras
for bar, media, desvio in zip(bars, medias, desvios):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + desvio,
             f'{media:.3f}s\n±{desvio:.3f}s',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# Gráfico 3: Distribuição de Tempos (Box Plot)
ax3 = axes[1, 0]
box_data = [tempos_spark, tempos_gemini, tempos_totais]
box = ax3.boxplot(box_data, labels=['Spark', 'Gemini', 'Total'],
                   patch_artist=True, notch=True, showmeans=True,
                   meanprops=dict(marker='D', markerfacecolor='red', markersize=8))

# Colorir as caixas
colors = ['#FF6B35', '#004E89', '#1B9E77']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax3.set_ylabel('Tempo (segundos)', fontsize=11, fontweight='bold')
ax3.set_title('Distribuição dos Tempos (Box Plot)', fontsize=12, fontweight='bold', pad=10)
ax3.grid(True, alpha=0.3, linestyle='--', axis='y')

# Gráfico 4: Coeficiente de Variação (Estabilidade)
ax4 = axes[1, 1]
cv_valores = [estatisticas['Spark']['Coef. Variação %'],
              estatisticas['Gemini']['Coef. Variação %'],
              estatisticas['Total']['Coef. Variação %']]

bars_cv = ax4.bar(categorias, cv_valores, 
                  color=['#FF6B35', '#004E89', '#1B9E77'], alpha=0.8, edgecolor='black', linewidth=1.5)
ax4.set_ylabel('Coeficiente de Variação (%)', fontsize=11, fontweight='bold')
ax4.set_title('Estabilidade da Performance (CV%)', fontsize=12, fontweight='bold', pad=10)
ax4.grid(True, alpha=0.3, linestyle='--', axis='y')
ax4.axhline(y=10, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Limite Aceitável (10%)')
ax4.legend(loc='upper right', fontsize=9)

# Adicionar valores nas barras
for bar, cv in zip(bars_cv, cv_valores):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
             f'{cv:.2f}%',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# ============================================
# RESUMO EXECUTIVO
# ============================================

print("\n\n" + "📊 " + "=" * 58)
print("📊 RESUMO EXECUTIVO")
print("=" * 60)

print(f"""
🔹 PERFORMANCE MÉDIA DO PIPELINE:
   • Conciliação Spark:      {estatisticas['Spark']['Média']:.4f}s ± {estatisticas['Spark']['Desvio Padrão']:.4f}s
   • Auditoria Gemini:       {estatisticas['Gemini']['Média']:.4f}s ± {estatisticas['Gemini']['Desvio Padrão']:.4f}s
   • Pipeline Completo:      {estatisticas['Total']['Média']:.4f}s ± {estatisticas['Total']['Desvio Padrão']:.4f}s

🔹 ESTABILIDADE (Coeficiente de Variação):
   • Spark:  {estatisticas['Spark']['Coef. Variação %']:.2f}% {'✅ Excelente' if estatisticas['Spark']['Coef. Variação %'] < 10 else '⚠️ Variável'}
   • Gemini: {estatisticas['Gemini']['Coef. Variação %']:.2f}% {'✅ Excelente' if estatisticas['Gemini']['Coef. Variação %'] < 10 else '⚠️ Variável'}
   • Total:  {estatisticas['Total']['Coef. Variação %']:.2f}% {'✅ Excelente' if estatisticas['Total']['Coef. Variação %'] < 10 else '⚠️ Variável'}

🔹 THROUGHPUT PROJETADO:
   • Cenários/segundo:      {500 / estatisticas['Total']['Média']:.2f}
   • Cenários/minuto:       {(500 / estatisticas['Total']['Média']) * 60:.0f}
   • Cenários/hora:         {(500 / estatisticas['Total']['Média']) * 3600:.0f}

🔹 CONCLUSÃO:
   O pipeline demonstrou {'alta' if max(cv_valores) < 15 else 'moderada'} estabilidade com tempo médio de 
   {estatisticas['Total']['Média']:.2f}s para processar 500 cenários. A etapa Gemini representa 
   {(estatisticas['Gemini']['Média'] / estatisticas['Total']['Média']) * 100:.1f}% do tempo total.
""")

print("=" * 60)
print("✅ ANÁLISE COMPLETA")
print("=" * 60)

In [0]:
# ============================================
# PERSISTÊNCIA DOS RESULTADOS PARA AUDITORIA
# ============================================

import socket
from datetime import datetime
from pyspark.sql import Row

print("\n💾 PERSISTINDO RESULTADOS PARA AUDITORIA")
print("=" * 60)

# Coletar metadata do ambiente
metadata_execucao = {
    "timestamp_execucao": datetime.now(),
    "databricks_runtime": os.environ.get("DATABRICKS_RUNTIME_VERSION", "N/A"),
    "spark_version": spark.version,
    "python_version": os.sys.version.split()[0],
    "modelo_gemini": "models/gemini-3.6-flash",
    "temperatura": 0.1,
    "num_execucoes": len(tempos_spark),
    "cenarios_por_execucao": 500,
    "usuario": spark.sql("SELECT current_user() as user").collect()[0]["user"],
    "notebook_path": dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
}

# ============================================
# TABELA 1: EXECUÇÕES INDIVIDUAIS
# ============================================

print("\n📊 [1/2] Criando tabela de execuções individuais...")

# Preparar dados das execuções
rows_execucoes = []
for resultado in resultados_execucoes:
    row = Row(
        timestamp_batch=metadata_execucao["timestamp_execucao"],
        execucao_id=resultado["execucao"],
        tempo_spark_s=float(resultado["tempo_spark"]),
        tempo_gemini_s=float(resultado["tempo_gemini"]),
        tempo_total_s=float(resultado["tempo_total"]),
        status_conformidade=resultado["status_conformidade"],
        acuracia_pct=float(resultado["acuracia_pct"]),
        modelo_gemini=metadata_execucao["modelo_gemini"],
        temperatura=metadata_execucao["temperatura"],
        cenarios_processados=metadata_execucao["cenarios_por_execucao"],
        databricks_runtime=metadata_execucao["databricks_runtime"],
        spark_version=metadata_execucao["spark_version"],
        python_version=metadata_execucao["python_version"],
        usuario=metadata_execucao["usuario"],
        notebook_path=metadata_execucao["notebook_path"]
    )
    rows_execucoes.append(row)

df_execucoes = spark.createDataFrame(rows_execucoes)

# Nome da tabela (ajuste o catalog e schema conforme sua convenção)
tabela_execucoes = "main.default.auditoria_performance_execucoes"

# Persistir com append (mantém histórico)
df_execucoes.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(tabela_execucoes)

print(f"   ✅ {len(rows_execucoes)} execuções salvas em: {tabela_execucoes}")

# ============================================
# TABELA 2: ESTATÍSTICAS AGREGADAS
# ============================================

print("\n📊 [2/2] Criando tabela de estatísticas agregadas...")

# Preparar dados das estatísticas
rows_stats = []
for etapa in ["Spark", "Gemini", "Total"]:
    row = Row(
        timestamp_batch=metadata_execucao["timestamp_execucao"],
        etapa=etapa,
        media_s=float(estatisticas[etapa]["Média"]),
        mediana_s=float(estatisticas[etapa]["Mediana"]),
        desvio_padrao_s=float(estatisticas[etapa]["Desvio Padrão"]),
        minimo_s=float(estatisticas[etapa]["Mínimo"]),
        maximo_s=float(estatisticas[etapa]["Máximo"]),
        coef_variacao_pct=float(estatisticas[etapa]["Coef. Variação %"]),
        num_execucoes=metadata_execucao["num_execucoes"],
        cenarios_por_execucao=metadata_execucao["cenarios_por_execucao"],
        throughput_cenarios_por_segundo=500 / estatisticas["Total"]["Média"] if etapa == "Total" else None,
        modelo_gemini=metadata_execucao["modelo_gemini"],
        temperatura=metadata_execucao["temperatura"],
        databricks_runtime=metadata_execucao["databricks_runtime"],
        usuario=metadata_execucao["usuario"],
        notebook_path=metadata_execucao["notebook_path"]
    )
    rows_stats.append(row)

df_stats = spark.createDataFrame(rows_stats)

tabela_stats = "main.default.auditoria_performance_estatisticas"

# Persistir com append (mantém histórico)
df_stats.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(tabela_stats)

print(f"   ✅ {len(rows_stats)} registros estatísticos salvos em: {tabela_stats}")

# ============================================
# HABILITAR TIME TRAVEL E AUDITORIA
# ============================================

print("\n🕒 Habilitando recursos de auditoria Delta Lake...")

# Habilitar Change Data Feed para rastreabilidade
spark.sql(f"ALTER TABLE {tabela_execucoes} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql(f"ALTER TABLE {tabela_stats} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print("   ✅ Change Data Feed habilitado em ambas as tabelas")

# ============================================
# RESUMO DA PERSISTÊNCIA
# ============================================

print("\n" + "=" * 60)
print("✅ DADOS PERSISTIDOS COM SUCESSO")
print("=" * 60)

print(f"""
📊 TABELAS CRIADAS:
   1. {tabela_execucoes}
      • Contém: Resultados individuais de cada execução
      • Linhas adicionadas: {len(rows_execucoes)}
      • Recursos: Delta Lake, Time Travel, Change Data Feed
      
   2. {tabela_stats}
      • Contém: Estatísticas agregadas por etapa
      • Linhas adicionadas: {len(rows_stats)}
      • Recursos: Delta Lake, Time Travel, Change Data Feed

🔍 CONSULTAS ÚTEIS PARA AUDITORIA:

   # Consultar última bateria de testes
   SELECT * FROM {tabela_execucoes}
   WHERE timestamp_batch = (SELECT MAX(timestamp_batch) FROM {tabela_execucoes})
   ORDER BY execucao_id
   
   # Evolução da performance ao longo do tempo
   SELECT 
     DATE(timestamp_batch) as data,
     etapa,
     AVG(media_s) as media_tempo_s,
     AVG(coef_variacao_pct) as media_cv_pct
   FROM {tabela_stats}
   GROUP BY DATE(timestamp_batch), etapa
   ORDER BY data DESC, etapa
   
   # Histórico de versões (Time Travel)
   DESCRIBE HISTORY {tabela_execucoes}
   
   # Auditar mudanças (Change Data Feed)
   SELECT * FROM table_changes('{tabela_execucoes}', 0)

🔒 CONFORMIDADE SOX:
   • Rastreabilidade completa: timestamp, usuário, ambiente
   • Imutabilidade: Delta Lake garante versionamento
   • Auditabilidade: Change Data Feed registra todas as operações
   • Reprodução: Metadata completa do ambiente e parâmetros
""")

print("=" * 60)

# 📊 Resumo Executivo - Bateria de Performance
## Pipeline de Auditoria Cognitiva SOX

---

## 🎯 Informações Gerais

| Item | Valor |
|------|-------|
| **Data/Hora da Execução** | 2026-09-06 20:09:11 UTC |
| **Número de Execuções** | 5 |
| **Cenários por Execução** | 500 |
| **Modelo Gemini** | gemini-3.6-flash |
| **Temperatura** | 0.1 (deterministico) |
| **Ambiente** | Databricks Serverless (Spark 4.2.0, Python 3.12.3) |

---

## ⚡ Performance Média do Pipeline

| Etapa | Tempo Médio | Desvio Padrão | Mínimo | Máximo | CV% |
|-------|-------------|---------------|--------|--------|-----|
| **Conciliação Spark** | 0.501s | ±0.076s | 0.404s | 0.616s | 15.2% |
| **Auditoria Gemini** | 7.559s | ±0.992s | 6.126s | 8.904s | 13.1% |
| **Pipeline Total** | 8.060s | ±0.988s | 6.631s | 9.410s | 12.3% |

### 📈 Throughput Projetado

* **62 cenários/segundo**
* **3.722 cenários/minuto**
* **223.327 cenários/hora**
* **5,36 milhões de cenários/dia** (operação 24/7)

---

## 📋 Resultados Individuais por Execução

| # | Spark | Gemini | Total | Status | Acurácia |
|---|-------|--------|-------|--------|----------|
| 1 | 0.616s | 7.483s | 8.099s | Em Risco Material | 85,4% |
| 2 | 0.477s | 7.467s | 7.943s | Em Risco Material | 85,4% |
| 3 | 0.504s | 6.126s | 6.631s | Em Risco Material | 85,4% |
| 4 | 0.506s | 8.904s | 9.410s | Em Risco Material | 85,4% |
| 5 | 0.404s | 7.813s | 8.217s | Em Risco Material | 85,4% |

---

## 🔍 Análise de Estabilidade

### Coeficiente de Variação (CV%)

* ✅ **CV < 10%**: Excelente estabilidade
* ⚠️ **CV 10-15%**: Variabilidade moderada
* ❌ **CV > 15%**: Alta variabilidade

| Etapa | CV% | Avaliação |
|-------|-----|----------|
| Spark | 15,2% | ⚠️ Variabilidade moderada |
| Gemini | 13,1% | ⚠️ Variabilidade moderada |
| Total | 12,3% | ⚠️ Variabilidade moderada |

### 🔎 Observações

1. **Spark (conciliação distribuída)**: Variação de 15,2% indica pequenas flutuações na alocação de recursos de rede/shuffle entre execuções
2. **Gemini (inferência LLM)**: Variação de 13,1% é típica de APIs de modelo generativo (fila, cold start, latência de rede)
3. **Gemini representa 93,8% do tempo total** → gargalo identificado

---

## 🎯 Diagnóstico de Conformidade

### Status Consolidado

**Todas as 5 execuções retornaram:**
* **Status**: Em Risco Material
* **Acurácia**: 85,4%
* **Conclusão**: Padrão consistente de não-conformidade detectado

### 🚨 Causa Raiz (Diagnóstico Gemini)

> "Falha sistemática no roteamento para conta transitória"

### ⚖️ Impacto SOX

* **Seção 404 (ICFR)**: Inoperância parcial dos controles
* **Risco de Distorção Material**: ALTO
* **Ação Requerida**: Correção imediata do motor de roteamento contábil

---

## 📊 Recomendações

### 1️⃣ Curto Prazo (Operacional)

* Investigar e corrigir a lógica de roteamento para conta transitória
* Executar validação pós-correção com nova bateria de 10+ execuções
* Meta de acurácia: ≥ 98%

### 2️⃣ Médio Prazo (Performance)

* **Otimizar Gemini (93,8% do tempo)**:
  * Avaliar modelo mais rápido (ex: gemini-1.5-flash-8b)
  * Implementar cache de respostas para padrões recorrentes
  * Considerar processamento em lote (batch inference)
* **Meta**: Reduzir tempo total de 8s → 4-5s

### 3️⃣ Longo Prazo (Governança)

* Estabelecer SLA de performance: P95 < 10s
* Implementar monitoramento contínuo de acurácia
* Criar alertas automáticos para CV% > 20%

---

## 🔐 Auditabilidade e Rastreabilidade

Todos os resultados foram persistidos nas seguintes tabelas Delta:

1. **`main.default.auditoria_performance_execucoes`**
   * Resultados individuais de cada execução
   * Metadata completa do ambiente
   * Change Data Feed habilitado

2. **`main.default.auditoria_performance_estatisticas`**
   * Estatísticas agregadas por etapa
   * Histórico temporal de performance
   * Time Travel disponível

### 📝 Consulta de Auditoria

```sql
-- Ver última bateria de testes
SELECT * 
FROM main.default.auditoria_performance_execucoes
WHERE timestamp_batch = (
  SELECT MAX(timestamp_batch) 
  FROM main.default.auditoria_performance_execucoes
)
ORDER BY execucao_id;
```

---

## ✅ Conclusão

O pipeline de auditoria cognitiva demonstrou:

* ✅ **Funcionalidade**: Opera consistentemente com 100% de sucesso nas 5 execuções
* ⚠️ **Performance**: Tempo médio de 8,06s com variabilidade moderada (CV 12,3%)
* ❌ **Conformidade**: Detectou risco material consistente (85,4% acurácia)
* ✅ **Escalabilidade**: Throughput de ~223k cenários/hora atende requisitos

**Próximo Passo Crítico**: Correção do motor de roteamento contábil antes de produção.

---

*Relatório gerado automaticamente pelo Pipeline de Auditoria Cognitiva SOX*  
*Databricks Assistant • 2026-09-06*